# Houston, TX parcels

Build a Houston city-only parcel export from the Harris County Appraisal District (HCAD)
public data downloads.

**Data sources:**

1. **Real property tabular data** — HCAD flat-file exports (tab-delimited ASCII):
   - Download: `https://download.hcad.org/data/CAMA/2025/Real_acct_owner.zip`
   - Contains: `real_acct.txt` with account numbers, appraised values, situs address,
     and mailing/owner info. **File has a header row.**
   - Other files in the ZIP (not used here): `owners.txt`, `deeds.txt`, `permits.txt`,
     `parcel_tieback.txt`, `real_mnrl.txt`, `real_neighborhood_code.txt`

2. **Parcel GIS data** — HCAD parcel shapefile (ESRI Shapefile format):
   - Download: `https://download.hcad.org/data/GIS/GIS_Public.zip`
   - Contains parcel polygons; join to tabular data via account number field.
   - Updated quarterly.

**Key real_acct.txt fields (verified from actual header row, 71 columns):**
- `acct` — 13-digit HCAD account number (parcel ID)
- `state_class` — Texas property state classification code
  (e.g. A1=single family, B1=multifamily, C1=vacant residential,
   F1=commercial, X=totally exempt, W=government/state owned)
- `tot_appr_val` — total appraised value
- `land_val` — appraised land value
- `bld_val` — appraised building/improvement value
- `x_features_val` — extra feature value (pools, sheds, etc.)
- `ag_val` — agricultural value
- `site_addr_1` — situs street address (e.g. "0 COMMERCE ST")
- `site_addr_2` — situs city (e.g. "HOUSTON")
- `site_addr_3` — situs ZIP code
- `mailto` — mailing entity name (owner name / entity); used for government heuristic
- `lgl_1`–`lgl_4` — legal description lines

**Note:** Detailed owner records are in `owners.txt` (not loaded here).

**Jurisdictional note:**
HCAD covers all of Harris County, which is much larger than the City of Houston.
We filter to the Houston city boundary using osmnx after joining.

**Parcel detail links:**
HCAD public search: `https://public.hcad.org/records/details.asp?crypt=<acct>`

**PMTiles:**
Houston has ~700,000+ parcels. Use PMTiles for performance.

In [ ]:
import io
import os
import glob
import zipfile
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime
from shapely.ops import unary_union
from shapely.geometry import MultiPolygon

import sys
sys.path.append('..')
from parcel_calculations import add_improvement_ratio_fields, classify_property_refined
from cloud_utils import ensure_geodataframe

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DOWNLOAD_DATA = 0   # set to 1 to download fresh data from HCAD
DATA_DIR = 'data/houston'
os.makedirs(DATA_DIR, exist_ok=True)

# Data year — HCAD releases data annually.
YEAR = 2025

# ---------------------------------------------------------------------------
# Download URLs (verified 2026-03)
# ---------------------------------------------------------------------------
# Real property tabular data
REAL_ACCT_URL = f'https://download.hcad.org/data/CAMA/{YEAR}/Real_acct_owner.zip'

# Parcel GIS shapefile
GIS_SHAPEFILE_URL = 'https://download.hcad.org/data/GIS/GIS_Public.zip'

## 1. Download or load cached HCAD data

### 1a. Real property tabular data (real_acct.txt)

In [ ]:
# real_acct.txt has a header row (verified from file) — 71 tab-delimited columns.
# Load with header=0 (default); no manual column list needed.
# Force string type for ID and ZIP fields to preserve leading zeros.
REAL_ACCT_DTYPES = {
    'acct':      str,
    'mail_zip':  str,
    'site_addr_3': str,  # situs ZIP
    'jurs':      str,
}


def download_and_extract_real_acct(url: str, data_dir: str) -> str:
    """Download Real_acct_owner.zip and extract real_acct.txt. Returns local path."""
    print(f'Downloading real property data from:\n  {url}')
    resp = requests.get(url, stream=True, timeout=300)
    resp.raise_for_status()
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    print(f'ZIP contents: {zf.namelist()}')
    real_acct_name = next(
        (n for n in zf.namelist() if n.lower().endswith('real_acct.txt')), None
    )
    if not real_acct_name:
        raise FileNotFoundError(
            f'real_acct.txt not found in ZIP. Contents: {zf.namelist()}'
        )
    out_path = os.path.join(data_dir, 'real_acct.txt')
    with zf.open(real_acct_name) as src, open(out_path, 'wb') as dst:
        dst.write(src.read())
    print(f'Extracted real_acct.txt → {out_path}')
    return out_path


real_acct_path = os.path.join(DATA_DIR, 'real_acct.txt')

if DOWNLOAD_DATA == 1 or not os.path.exists(real_acct_path):
    real_acct_path = download_and_extract_real_acct(REAL_ACCT_URL, DATA_DIR)
else:
    print(f'Using cached real_acct.txt: {real_acct_path}')

# Load — file has a header row, so header=0 (default)
print('Loading real_acct.txt...')
real_acct_df = pd.read_csv(
    real_acct_path,
    sep='\t',
    header=0,
    dtype=REAL_ACCT_DTYPES,
    low_memory=False,
    on_bad_lines='warn',
    encoding='latin-1',
)
print(f'Loaded real_acct: {len(real_acct_df):,} rows × {len(real_acct_df.columns)} cols')
print('Columns:', real_acct_df.columns.tolist())
print(real_acct_df.head(2))

### 1b. GIS parcel shapefile

In [ ]:
def download_and_extract_shapefile(url: str, data_dir: str) -> str:
    """
    Download GIS_Public.zip, extract Parcels.zip from it, then extract the shapefile.

    GIS_Public.zip structure (verified 2026-03):
      GIS_Public.zip
        Abstract.zip, Blk_num.zip, City.zip, ... Parcels.zip ...
      Parcels.zip
        Parcels.shp, Parcels.dbf, Parcels.shx, ...

    CRS of Parcels.shp: EPSG:2278 (Texas State Plane South Central, US feet).
    Will be reprojected to EPSG:4326 before spatial operations.
    """
    shp_dir = os.path.join(data_dir, 'parcels_shp')
    os.makedirs(shp_dir, exist_ok=True)
    print(f'Downloading parcel shapefile from:\n  {url}')
    resp = requests.get(url, stream=True, timeout=600)
    resp.raise_for_status()

    outer_zf = zipfile.ZipFile(io.BytesIO(resp.content))
    outer_names = outer_zf.namelist()
    print(f'Outer ZIP contains {len(outer_names)} entries, e.g.: {outer_names[:6]}')

    # GIS_Public.zip contains many inner ZIPs; we want Parcels.zip
    parcels_zip_name = next(
        (n for n in outer_names if n.lower().endswith('parcels.zip')), None
    )
    if parcels_zip_name:
        print(f'Extracting inner ZIP: {parcels_zip_name}')
        inner_data = outer_zf.read(parcels_zip_name)
        inner_zf = zipfile.ZipFile(io.BytesIO(inner_data))
        print(f'Parcels.zip contents: {inner_zf.namelist()}')
        inner_zf.extractall(shp_dir)
    else:
        # Fallback: extract everything and search
        print('Parcels.zip not found in outer ZIP — extracting all and searching for .shp')
        outer_zf.extractall(shp_dir)

    print(f'Extracted to {shp_dir}')
    return shp_dir


shp_dir = os.path.join(DATA_DIR, 'parcels_shp')
shp_files = glob.glob(os.path.join(shp_dir, '**', '*.shp'), recursive=True)

if DOWNLOAD_DATA == 1 or not shp_files:
    shp_dir = download_and_extract_shapefile(GIS_SHAPEFILE_URL, DATA_DIR)
    shp_files = glob.glob(os.path.join(shp_dir, '**', '*.shp'), recursive=True)

if not shp_files:
    raise FileNotFoundError(
        f'No .shp files found in {shp_dir}.\n'
        'Manually extract Parcels.zip from GIS_Public.zip and place Parcels.shp '
        f'under {shp_dir}, then re-run with DOWNLOAD_DATA=0.'
    )

shp_path = shp_files[0]
print(f'Loading shapefile: {shp_path}')
parcel_gdf = gpd.read_file(shp_path)
print(f'Loaded shapefile | CRS={parcel_gdf.crs} | rows={len(parcel_gdf):,}')
print('Shapefile columns:', parcel_gdf.columns.tolist())

## 2. Inspect raw fields

In [ ]:
pd.set_option('display.max_columns', None)
display(parcel_gdf.head(3))
display(real_acct_df.head(3))

In [ ]:
# HCAD shapefile account number field: HCAD_NUM (verified from actual file)
# Also available in shapefile (no join needed):
#   city      — situs city (more reliable than site_addr_2 from tabular)
#   zip       — situs ZIP
#   CurrOwner — current owner name (used for government exemption heuristic)
#   Stacked   — 1 if stacked/condo parcel, 0 otherwise
#   LocAddr   — location address string
#   StatedArea, Acreage — area fields (shapefile-native)
SHP_ACCT_CANDIDATES = [
    'HCAD_NUM', 'hcad_num', 'ACCT', 'acct', 'ACCOUNT', 'account',
    'HCAD_ACCT', 'hcad_acct', 'PARCEL_ID', 'parcel_id',
]
shp_acct_col = next(
    (c for c in SHP_ACCT_CANDIDATES if c in parcel_gdf.columns), None
)
if shp_acct_col is None:
    print('Could not identify account column in shapefile.')
    print('Available columns:', parcel_gdf.columns.tolist())
    raise KeyError(
        'Update SHP_ACCT_CANDIDATES above to match the account field in your shapefile.'
    )
print(f'Shapefile account column: {shp_acct_col}')

# Normalize account number to zero-padded 13-character string in both tables
parcel_gdf[shp_acct_col] = parcel_gdf[shp_acct_col].astype(str).str.strip().str.zfill(13)
real_acct_df['acct'] = real_acct_df['acct'].astype(str).str.strip().str.zfill(13)

## 3. Join tabular property data to parcel geometry

In [ ]:
# Select only the columns we need from real_acct for the join.
# Actual column names verified from file header row (2025 data).
#   land_val        = appraised land value
#   bld_val         = building / improvement value
#   x_features_val  = extra features (pools, fences, etc.)
#   ag_val          = agricultural value
#   tot_appr_val    = total appraised value
#   bld_ar          = building area (sqft) — vacancy signal for refined classification
#   site_addr_2     = situs city  (e.g. "HOUSTON")
#   mailto          = owner/entity mailing name; used as government exemption proxy
KEEP_COLS = [
    'acct', 'mailto', 'site_addr_2', 'state_class',
    'tot_appr_val', 'land_val', 'bld_val', 'x_features_val', 'ag_val', 'bld_ar',
]
keep_cols_present = [c for c in KEEP_COLS if c in real_acct_df.columns]
missing = set(KEEP_COLS) - set(keep_cols_present)
if missing:
    print(f'WARNING: Missing expected columns from real_acct: {missing}')
    print('Available:', real_acct_df.columns.tolist())

real_acct_slim = real_acct_df[keep_cols_present].copy()

for col in ['tot_appr_val', 'land_val', 'bld_val', 'x_features_val', 'ag_val', 'bld_ar']:
    if col in real_acct_slim.columns:
        real_acct_slim[col] = pd.to_numeric(real_acct_slim[col], errors='coerce')

print(
    f'Joining {len(parcel_gdf):,} parcel geometries '
    f'with {len(real_acct_slim):,} tabular records on account number...'
)

parcel_gdf = parcel_gdf.merge(
    real_acct_slim,
    left_on=shp_acct_col,
    right_on='acct',
    how='left',
)

if shp_acct_col != 'acct':
    parcel_gdf['acct'] = parcel_gdf[shp_acct_col]

print(f'Joined | rows={len(parcel_gdf):,}')
print(f'Rows with no tabular match: {parcel_gdf["acct"].isna().sum():,}')
print(f'Rows with missing tot_appr_val: {parcel_gdf["tot_appr_val"].isna().sum():,}')

## 4. Inspect state_class distribution

In [ ]:
if 'state_class' in parcel_gdf.columns:
    print('state_class value counts:')
    with pd.option_context('display.max_rows', 80):
        print(parcel_gdf['state_class'].value_counts(dropna=False).head(80))
else:
    print('⚠️  state_class column not found after join. Available columns:')
    print(list(parcel_gdf.columns))

## 5. Restrict to Houston city boundary

HCAD covers all of Harris County. Houston city limits are irregular and extend
in multiple directions — a bounding box would capture many non-Houston parcels.
We use the authoritative OSM city boundary polygon via osmnx.

In [ ]:
import osmnx as ox

# Parcels.shp uses EPSG:2278 (Texas State Plane South Central, US feet).
# Reproject to EPSG:4326 before spatial operations with OSM boundary.
if parcel_gdf.crs is None:
    print('WARNING: shapefile has no CRS; assuming EPSG:2278')
    parcel_gdf = parcel_gdf.set_crs('EPSG:2278')

if parcel_gdf.crs.to_epsg() != 4326:
    parcel_gdf = parcel_gdf.to_crs('EPSG:4326')
    print(f'Reprojected parcels from {parcel_gdf.crs} → EPSG:4326')

print('Fetching Houston city boundary from OSM...')
houston_boundary = ox.geocode_to_gdf('Houston, Texas, USA')
print(f'Boundary CRS: {houston_boundary.crs}')

if houston_boundary.crs != parcel_gdf.crs:
    houston_boundary = houston_boundary.to_crs(parcel_gdf.crs)

boundary_geom = houston_boundary.geometry.iloc[0]
print(f'Houston boundary type: {boundary_geom.geom_type}')

In [ ]:
# Pre-filter on the shapefile's `city` column (verified field name from actual file).
# This is more reliable than site_addr_2 from the tabular data.
# Keeps rows where city is HOUSTON or is null (GIS record may lack city for some parcels).
if 'city' in parcel_gdf.columns:
    n_before_city = len(parcel_gdf)
    parcel_gdf = parcel_gdf[
        parcel_gdf['city'].isna() |
        parcel_gdf['city'].str.upper().str.contains('HOUSTON', na=False)
    ].copy()
    print(
        f'City pre-filter: {n_before_city:,} → {len(parcel_gdf):,} '
        f'(removed {n_before_city - len(parcel_gdf):,} clearly non-Houston rows)'
    )

# Spatial filter: keep parcels whose centroid falls within the Houston boundary
print('Applying spatial filter to Houston boundary...')
n_before = len(parcel_gdf)

valid_geom_mask = (
    parcel_gdf['geometry'].notnull() &
    parcel_gdf['geometry'].apply(lambda g: getattr(g, 'is_valid', False))
)
centroids = parcel_gdf.loc[valid_geom_mask, 'geometry'].centroid
inside_mask = valid_geom_mask.copy()
inside_mask[valid_geom_mask] = centroids.within(boundary_geom)

parcel_gdf = parcel_gdf[inside_mask].copy()
n_after = len(parcel_gdf)
print(f'Before: {n_before:,}  |  After: {n_after:,}  |  Removed: {n_before - n_after:,}')
print(f'Bounds: {parcel_gdf.total_bounds}')

In [ ]:
# Sanity check: bounds should be roughly:
#   minx ~ -95.9  maxx ~ -95.0  miny ~ 29.5  maxy ~ 30.1
bounds = parcel_gdf.total_bounds
assert -96.5 < bounds[0] < -95.0, f'Unexpected minx: {bounds[0]}'
assert -95.5 < bounds[2] < -94.5, f'Unexpected maxx: {bounds[2]}'
assert  29.0 < bounds[1] < 29.8,  f'Unexpected miny: {bounds[1]}'
assert  29.8 < bounds[3] < 30.3,  f'Unexpected maxy: {bounds[3]}'
print('✅ Bounds sanity check passed')

## 6. Handle duplicate account numbers (condos, multi-record parcels)

In [ ]:
acct_col = 'acct'
n_dupes = parcel_gdf.duplicated(subset=[acct_col], keep=False).sum()
print(f'Duplicate rows by {acct_col}: {n_dupes:,}')

if n_dupes > 0:
    print('Collapsing duplicates: summing value fields, unioning geometries...')

    numeric_sum_cols = [
        c for c in ['tot_appr_val', 'tot_land_val', 'tot_bldg_val', 'tot_extra_features_val']
        if c in parcel_gdf.columns
    ]
    categorical_cols = [
        c for c in parcel_gdf.columns
        if c not in set(numeric_sum_cols + ['geometry', acct_col])
    ]

    agg_dict = {c: 'sum' for c in numeric_sum_cols}
    agg_dict.update({c: 'first' for c in categorical_cols})

    collapsed = (
        parcel_gdf.groupby(acct_col, dropna=False).agg(agg_dict).reset_index()
    )
    geom_union = parcel_gdf.groupby(acct_col, dropna=False)['geometry'].apply(
        lambda geoms: unary_union([g for g in geoms if g is not None])
        if any(g is not None for g in geoms) else None
    )
    collapsed['geometry'] = geom_union.values
    parcel_gdf = gpd.GeoDataFrame(collapsed, geometry='geometry', crs=parcel_gdf.crs)
    print(f'✅ Rows after duplicate collapse: {len(parcel_gdf):,}')
else:
    print('✅ No duplicates — skipping collapse step')

## 7. Classify property type using Texas state codes

Texas property state classification codes (`state_class`) are standardized statewide.
Reference: Texas Property Tax Code, Comptroller Property Classification Manual.

Key codes:
- **A1, A2** — Single-family residential
- **B1, B2, B3, B4** — Multifamily residential (apartments, duplexes)
- **C1, C2, C3** — Vacant residential land
- **D1** — Qualified agricultural land (productivity appraisal)
- **D2** — Farm and ranch improvements on qualified ag land
- **E1, E2, E3** — Farm and ranch, improved
- **F1** — Commercial real property
- **F2** — Industrial real property
- **G1** — Gas, oil, casinghead gas
- **J** — Utilities (J1–J8)
- **L1** — Commercial personal property (inventory)
- **L2** — Industrial personal property (inventory)
- **M1** — Tangible personal property / mobile homes
- **O1** — Residential inventory (builder spec homes)
- **S** — Special inventory
- **W** — Assessed for ownership/control only (government, state)
- **X** — Totally exempt property
- **U** — Utility personal property

In [ ]:
def categorize_property_type(state_class_val):
    """
    Assigns a human-readable property category from the Texas state_class field.
    """
    raw = str(state_class_val or '').strip().upper()

    if raw in ('X',):                             return 'Exempt'
    if raw in ('W',):                             return 'Government / State'
    if raw in ('A1', 'A2'):                       return 'Single Family'
    if raw in ('B1', 'B2', 'B3', 'B4'):           return 'Multifamily'
    if raw in ('C1', 'C2', 'C3'):                 return 'Vacant Residential'
    if raw in ('D1', 'D2', 'E1', 'E2', 'E3'):     return 'Agricultural / Rural'
    if raw == 'F1':                               return 'Commercial'
    if raw == 'F2':                               return 'Industrial'
    if raw.startswith('G'):                       return 'Mineral / Oil & Gas'
    if raw.startswith('J') or raw.startswith('U'): return 'Utility'
    if raw in ('L1', 'L2', 'M1', 'O1', 'S'):     return 'Personal Property / Inventory'
    return 'Other'


if 'state_class' in parcel_gdf.columns:
    parcel_gdf['PROPERTY_CATEGORY'] = parcel_gdf['state_class'].apply(categorize_property_type)
else:
    parcel_gdf['PROPERTY_CATEGORY'] = 'Other'

with pd.option_context('display.max_rows', None):
    print('PROPERTY_CATEGORY value counts:')
    print(parcel_gdf['PROPERTY_CATEGORY'].value_counts(dropna=False))

## 8. Exclude exempt parcels

Exempt parcels have distorted or zero assessed values and must be excluded.

Houston-specific exemption sources:
- `state_class == 'X'` — totally exempt (churches, schools, nonprofits, government)
- `state_class == 'W'` — government/state ownership
- Derived: owner name contains known government keywords
  (some government parcels are miscoded with non-exempt state codes)

Note: Homestead exemptions do NOT make a parcel fully exempt — they only reduce
the taxable value. Do not exclude parcels solely on homestead exemption.

In [ ]:
export_gdf = parcel_gdf.copy()

# Exempt by state_class
exempt_by_state = export_gdf['PROPERTY_CATEGORY'].isin(['Exempt', 'Government / State'])

# Exempt by ownership keyword heuristic.
# `CurrOwner` is available directly from the shapefile (verified field name).
# Fall back to `mailto` from real_acct.txt if CurrOwner is absent.
GOVT_KEYWORDS = [
    'CITY OF HOUSTON', 'HARRIS COUNTY', 'STATE OF TEXAS', 'HISD',
    'HOUSTON ISD', 'HARRIS CTY', 'METRO', 'PORT OF HOUSTON',
    'UNITED STATES', 'US GOVT', 'U.S. GOVERNMENT',
]

keyword_pattern = '|'.join(GOVT_KEYWORDS)

if 'CurrOwner' in export_gdf.columns:
    owner_str = export_gdf['CurrOwner'].astype(str).str.upper()
    exempt_by_ownership = owner_str.str.contains(keyword_pattern, na=False)
    print(f'Exempt by ownership keyword (CurrOwner): {exempt_by_ownership.sum():,}')
elif 'mailto' in export_gdf.columns:
    owner_str = export_gdf['mailto'].astype(str).str.upper()
    exempt_by_ownership = owner_str.str.contains(keyword_pattern, na=False)
    print(f'Exempt by ownership keyword (mailto fallback): {exempt_by_ownership.sum():,}')
else:
    exempt_by_ownership = pd.Series(False, index=export_gdf.index)
    print('WARNING: neither CurrOwner nor mailto found — skipping ownership exemption heuristic')

export_gdf['exemption_flag'] = (exempt_by_state | exempt_by_ownership).astype(int)
print('Exemption flag counts:')
print(export_gdf['exemption_flag'].value_counts())

before = len(export_gdf)
export_gdf = export_gdf[export_gdf['exemption_flag'] == 0].copy()
print(f'Removed {before - len(export_gdf):,} fully exempt parcels')
print(f'Rows remaining: {len(export_gdf):,}')

## 9. Refined land use classification

In [ ]:
export_gdf['property_land_use_category'] = export_gdf['PROPERTY_CATEGORY']

# Drop non-geographic categories before export
non_geo_cats = {'Mineral / Oil & Gas', 'Personal Property / Inventory'}
before = len(export_gdf)
export_gdf = export_gdf[~export_gdf['property_land_use_category'].isin(non_geo_cats)].copy()
print(f'Dropped {before - len(export_gdf):,} non-geographic records (minerals, personal property)')

# Use actual column names from real_acct.txt: land_val, bld_val
export_gdf['land_value']        = pd.to_numeric(export_gdf.get('land_val', np.nan), errors='coerce')
export_gdf['improvement_value'] = pd.to_numeric(export_gdf.get('bld_val',  np.nan), errors='coerce')

# Refined classification (Vacant / Underdeveloped / Parking Lot) using every available
# signal — see classify_property_refined() in parcel_calculations.py. Houston calibration:
#   - Single Family is flagged Underdeveloped only at land/total >= 0.67 (vs 0.50
#     elsewhere): inner-loop land routinely outvalues a modest house, and a flat 0.50
#     over-flags ~half the city's homes.
#   - A no-improvement-value parcel becomes Vacant only if it ALSO has no building sqft
#     (bld_ar), no Overture building footprint, and isn't exempt/utility/ag (state_class
#     X/J/D). That keeps parks, ROW and unvalued-but-built structures from being flagged.
# NOTE: this fetches Overture building footprints (DuckDB + network) for the
# no-improvement-value candidates only, so the cell needs internet access when it runs.
export_gdf['property_land_use_refined'] = classify_property_refined(export_gdf)

print('Refined category counts:')
print(export_gdf['property_land_use_refined'].value_counts(dropna=False))

## 10. Compute canonical fields

In [ ]:
from pyproj import Geod

geod = Geod(ellps='WGS84')


def geodesic_area_sqft(geom):
    if geom is None or geom.is_empty:
        return np.nan
    gtype = geom.geom_type
    if gtype == 'Polygon':
        lon, lat = geom.exterior.coords.xy
        area_m2, _ = geod.polygon_area_perimeter(lon, lat)
        return abs(area_m2) * 10.763910416709722
    if gtype == 'MultiPolygon':
        return sum(geodesic_area_sqft(p) for p in geom.geoms)
    return np.nan


# Validate and repair geometries
export_gdf['geometry'] = export_gdf['geometry'].apply(
    lambda g: g if g is None or g.is_valid else g.buffer(0)
)

print('Computing parcel areas...')
export_gdf['area_sqft'] = export_gdf['geometry'].apply(geodesic_area_sqft)
export_gdf.loc[export_gdf['area_sqft'] < 1, 'area_sqft'] = np.nan

# Total appraised value
if 'tot_appr_val' in export_gdf.columns:
    export_gdf['full_market_value'] = pd.to_numeric(export_gdf['tot_appr_val'], errors='coerce')
else:
    export_gdf['full_market_value'] = (
        export_gdf.get('land_val', pd.Series(0)).fillna(0)
        + export_gdf.get('bld_val', pd.Series(0)).fillna(0)
    )

# Land and improvement values — actual column names from real_acct.txt
export_gdf['land_value']        = pd.to_numeric(export_gdf.get('land_val', np.nan), errors='coerce')
export_gdf['improvement_value'] = pd.to_numeric(export_gdf.get('bld_val',  np.nan), errors='coerce')

# Per-sqft metrics
export_gdf['full_market_value_per_sqft']  = export_gdf['full_market_value']  / export_gdf['area_sqft']
export_gdf['land_value_per_sqft']         = export_gdf['land_value']          / export_gdf['area_sqft']
export_gdf['improvement_value_per_sqft']  = export_gdf['improvement_value']   / export_gdf['area_sqft']

# Improvement / land ratio derived fields
export_gdf = add_improvement_ratio_fields(
    export_gdf,
    land_col='land_value',
    improvement_col='improvement_value',
)

print('Canonical fields computed')

## 11. Build parcel detail link (HCAD public search)

In [ ]:
# Parcel detail link.
# NOTE: HCAD's record page (public.hcad.org/records/details.asp?crypt=...) requires an
# ENCRYPTED 'crypt' token, NOT the raw account number — passing the account 404s. There
# is no public direct-by-account deep-link, so we point at HCAD's search portal with the
# account carried in the URL. (The web client also recovers the account from older
# crypt-style baked links, so already-deployed tiles keep working without a re-bake.)
export_gdf['link'] = (
    'https://search.hcad.org/?account='
    + export_gdf['acct'].astype(str).str.strip()
)
print('Sample links:')
print(export_gdf['link'].head())

## 12. Select and export canonical parquet

In [ ]:
COLUMNS_TO_EXPORT = [
    'geometry',
    'exemption_flag',
    'property_land_use_category',
    'property_land_use_refined',
    'full_market_value',
    'full_market_value_per_sqft',
    'land_value',
    'land_value_per_sqft',
    'improvement_value',
    'improvement_value_per_sqft',
    'TLLDIMPROV',
    'IMPR_LAND_RATIO',
    'IMPR_LAND_PCT',
    'IMPR_PCT_TOTAL',
    'link',
]

# Guarantee all columns present (fill missing with NaN)
for col in COLUMNS_TO_EXPORT:
    if col not in export_gdf.columns:
        export_gdf[col] = np.nan

export_final = export_gdf[COLUMNS_TO_EXPORT].rename(columns={
    'land_value': 'current_full_land_value'
})

# Final geometry validation
export_final['geometry'] = export_final['geometry'].apply(
    lambda g: g if g is None or g.is_valid else g.buffer(0)
)

# Ensure CRS is EPSG:4326
export_final = gpd.GeoDataFrame(export_final, geometry='geometry', crs=export_gdf.crs)
if export_final.crs is None or export_final.crs.to_epsg() != 4326:
    export_final = export_final.to_crs('EPSG:4326')
    print('✅ Converted to EPSG:4326')

# Save canonical + dated parquets
canonical_path = os.path.join(DATA_DIR, 'houston-tx-parcels.parquet')
today_str = datetime.now().strftime('%Y_%m_%d')
dated_path = os.path.join(DATA_DIR, f'houston-tx-parcels_{today_str}.parquet')

export_final.to_parquet(canonical_path, index=False)
export_final.to_parquet(dated_path, index=False)

print(f'✅ Saved canonical parquet: {canonical_path}')
print(f'✅ Saved dated parquet:     {dated_path}')
print('Export columns:', export_final.columns.tolist())
print(f'Total rows exported: {len(export_final):,}')
print('\nRefined category counts:')
print(export_final['property_land_use_refined'].value_counts(dropna=False))
print('\nGeometry type counts:')
print(export_final.geometry.geom_type.value_counts())
print('\nBounds:', export_final.total_bounds)

## 13. Upload parquet to dev Azure blob

In [ ]:
upload_dev = True

if upload_dev:
    from azure.storage.blob import BlobServiceClient

    connection_string = os.getenv('AZURE_STORAGE_CONNECTION_STRING')
    if not connection_string:
        raise ValueError(
            'Set AZURE_STORAGE_CONNECTION_STRING before uploading.'
        )

    container = os.getenv('AZURE_DEV_CONTAINER', 'parquets-dev')
    blob_name = 'houston-tx-parcels.parquet'
    local_path = os.path.join(DATA_DIR, blob_name)

    if not os.path.exists(local_path):
        raise FileNotFoundError(f'Local parquet not found: {local_path}')

    blob_service = BlobServiceClient.from_connection_string(connection_string)
    container_client = blob_service.get_container_client(container)

    with open(local_path, 'rb') as handle:
        container_client.upload_blob(name=blob_name, data=handle, overwrite=True)

    print(f'✅ Uploaded {local_path} -> {container}/{blob_name}')
else:
    print('upload_dev is False; skipping upload.')

## 14. Generate PMTiles and upload to dev

Houston has 700,000+ parcels — PMTiles is required for acceptable performance.
Run this after the parquet has been validated locally.

In [ ]:
upload_dev_pmtiles = True  # set False to skip PMTiles generation

if upload_dev_pmtiles:
    import platform
    import subprocess
    from pathlib import Path

    notebook_dir = Path.cwd()
    current = notebook_dir
    project_root = None
    while current.parent != current:
        if (current / 'data' / 'scripts' / 'parquet_to_pmtiles.py').exists():
            project_root = current
            break
        current = current.parent

    if not project_root:
        project_root = (
            notebook_dir.parent.parent
            if notebook_dir.name == 'jurisidictions'
            else notebook_dir.parent
        )

    script_path = project_root / 'data' / 'scripts' / 'parquet_to_pmtiles.py'
    if not script_path.exists():
        raise FileNotFoundError(f'parquet_to_pmtiles.py not found at {script_path}')

    cmd = [sys.executable, str(script_path), '--city', 'houston', '--upload', '--overwrite']

    if not os.getenv('AZURE_STORAGE_CONNECTION_STRING'):
        print('WARNING: AZURE_STORAGE_CONNECTION_STRING not set — running without upload.')
        cmd.remove('--upload')

    # On Windows, tippecanoe has no native binary — route through WSL2.
    # The script auto-detects WSL if tippecanoe is missing, but --wsl makes it explicit.
    # Prerequisites (one-time, in a WSL2 terminal):
    #   wsl -- sudo apt-get update && sudo apt-get install -y tippecanoe
    #   wsl -- sudo apt-get install -y pmtiles  # or: pip install pmtiles
    if platform.system() == 'Windows':
        cmd.append('--wsl')
        print('Windows detected: adding --wsl flag (tippecanoe will run via WSL2)')

    print(f'Running PMTiles conversion: {" ".join(cmd)}')
    result = subprocess.run(cmd, cwd=str(project_root), capture_output=True, text=True)

    if result.returncode == 0:
        print('PMTiles conversion and upload completed!')
        print('houston-tx-parcels.pmtiles')
        print('houston-tx-parcels-metadata.json')
        if result.stdout:
            print(result.stdout)
    else:
        print(f'PMTiles conversion failed (exit {result.returncode})')
        if result.stderr:
            print(result.stderr)
        if result.stdout:
            print(result.stdout)
else:
    print('upload_dev_pmtiles is False; skipping PMTiles generation.')

## 15. Promote to prod

Only run after validating the dev deployment at:
- `/app.html?city=houston`

Artifacts to promote:
- `houston-tx-parcels.parquet`
- `houston-tx-parcels.pmtiles`
- `houston-tx-parcels-metadata.json`

In [ ]:
promote_to_prod = False  # set True only after dev validation
promote_overwrite = True

PROD_ARTIFACTS = [
    'houston-tx-parcels.parquet',
    'houston-tx-parcels.pmtiles',
    'houston-tx-parcels-metadata.json',
]

if promote_to_prod:
    from azure.storage.blob import BlobServiceClient

    connection_string = os.getenv('AZURE_STORAGE_CONNECTION_STRING')
    if not connection_string:
        raise ValueError('Set AZURE_STORAGE_CONNECTION_STRING before promotion.')

    dev_container  = os.getenv('AZURE_DEV_CONTAINER',  'parquets-dev')
    prod_container = os.getenv('AZURE_PROD_CONTAINER', 'parquets-prod')
    blob_service   = BlobServiceClient.from_connection_string(connection_string)

    for blob_name in PROD_ARTIFACTS:
        dev_blob  = blob_service.get_blob_client(dev_container,  blob_name)
        prod_blob = blob_service.get_blob_client(prod_container, blob_name)

        if not dev_blob.exists():
            print(f'⚠️  Dev blob not found, skipping: {dev_container}/{blob_name}')
            continue

        if prod_blob.exists():
            if not promote_overwrite:
                raise FileExistsError(
                    f'Prod blob already exists. Set promote_overwrite=True: {prod_container}/{blob_name}'
                )
            prod_blob.delete_blob()

        prod_blob.start_copy_from_url(dev_blob.url)
        print(f'✅ Promoted {dev_container}/{blob_name} -> {prod_container}/{blob_name}')
else:
    print('promote_to_prod is False; skipping prod promotion.')